# Bayan: Colab runner
Runs the whole pipeline end-to-end (T4 GPU runtime recommended: *Runtime > Change runtime type > T4 GPU*).
Stages are idempotent scripts under `scripts/`; the frozen test set is touched only by stage 8 (`s08_final.py`, guarded by a lock file).

**Before running:** put the course `Data` folder somewhere in your Drive and set `DATA_DIR` below.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, subprocess
REPO = '/content/bayan_project'   # clone your GitHub repo here, or unzip the project
DATA_DIR = '/content/drive/MyDrive/SDAIA NLP/NATURAL LANGUAGE PROCESSING WITH TRANSFORMER/Data'  # <- edit if your Data folder lives elsewhere
WORK = '/content/bayan_work'
os.environ.update(PYTHONPATH=f'{REPO}/src', BAYAN_DATA=DATA_DIR, BAYAN_WORK=WORK, BAYAN_REPORTS=f'{REPO}/reports', PYTHONUNBUFFERED='1')
print(os.listdir(DATA_DIR))

In [ ]:
!cd {REPO} && pip install -q -r requirements.txt && python -m pytest -q tests

## 1. Data audit, frozen splits, TF-IDF baselines, tokenizer audit

In [ ]:
!cd {REPO} && python scripts/s01_prepare.py

## 2. Classifiers (topic + sentiment), NER, QA  (about 40-60 min on a T4)

In [ ]:
!cd {REPO} && bash scripts/run_s02.sh

In [ ]:
!cd {REPO} && bash scripts/run_s03_s04.sh && python scripts/s03_train_ner.py --name ner_xlmr_clitic_aug --clitic_aug 0.35

In [ ]:
!cd {REPO} && python scripts/s02_train_cls.py --task topic --name topic_xlmr_all_ca --ckpt xlm-roberta-base --lang all --profile xlmr --epochs 4 --lr 3e-5 --bs 32 --case_aug 0.35

## 3. Search (bi-encoders, FAISS index, stage-2 anchor re-scorer, no-answer threshold)

In [ ]:
!cd {REPO} && python scripts/s05_search.py

## 4. Distilled students + Gulf-adapted variant

In [ ]:
!cd {REPO} && python scripts/s06b_distill.py --teacher topic_xlmr_all_ca --layers 2,5,8,11 --name topic_xlmr_l4_kd && python scripts/s06b_distill.py --teacher topic_xlmr_all_ca --layers 3,7,11 --name topic_xlmr_l3_kd && python scripts/s02c_gulf_adapt.py --base topic_xlmr_all_ca

## 5. Selection on VALIDATION, behavioural suite, validation error sheet

In [ ]:
!cd {REPO} && python scripts/s07_select.py && python scripts/s07b_behaviour_errors.py

## 6. Ship the 3-layer student, optimisation ladder for every classifier, quantisation decisions, HTTP benchmark (run on an otherwise idle CPU, >= 2 cores)

In [ ]:
!cd {REPO} && python scripts/s07e_finalize_config.py && for m in topic_xlmr_all_ca topic_xlmr_l4_kd topic_xlmr_l3_kd; do python scripts/s06_optimize.py --model $m || break; done && python scripts/s06c_quant_decisions.py

In [ ]:
!cd {REPO} && python scripts/s07c_http_bench.py --precision int8 --pin --threads 1 --max-wait-ms 0 --seconds 25 --think-ms-list 0,250,500,1000 --tag final_int8 && python scripts/s07c_http_bench.py --precision fp32 --pin --threads 1 --max-wait-ms 0 --seconds 25 --think-ms-list 500,1000 --tag final_fp32 && python scripts/s10_error_analysis.py

## 7. The ONE frozen-test evaluation, then documents

In [ ]:
!cd {REPO} && python scripts/s08_final.py && python scripts/s09_docs.py

## 8. Serve
`make serve` (INT8 default) or `make serve-fp32` (rollback); skew canaries run at start-up and the service refuses to start if they fail.

In [ ]:
!cd {REPO} && BAYAN_ARTIFACTS={WORK}/artifacts BAYAN_MODELS={WORK}/models timeout 60 python -m uvicorn bayan.service:app_factory --factory --port 8000 &